In [1]:
!pip install scikit-image

In [2]:
import sys
import os

# This tells the notebook to look one folder up (..) for the 'src' folder
sys.path.append(os.path.abspath('..'))

from src.rules import check_missing_fields, detect_font_inconsistency
from src.preprocessing import extract_ocr_from_xml

# 1. Run checks on Validation Split
# Using raw string (r"") to handle Windows backslashes
val_path = "../data/val"

# 2. Identify files that have a matching XML
# This ensures we only loop through valid pairs
xml_files = [f.replace('.xml', '') for f in os.listdir(val_path) if f.endswith('.xml')]

predictions = []

print(f"Found {len(xml_files)} annotated pairs to process in {val_path}...")

for xml_base in xml_files:
    # Construct the XML path
    xml_path = os.path.join(val_path, xml_base + ".xml")
    
    # Check for image file (handles both .png and .jpg)
    img_path_png = os.path.join(val_path, xml_base + ".png")
    img_path_jpg = os.path.join(val_path, xml_base + ".jpg")
    img_path = img_path_png if os.path.exists(img_path_png) else img_path_jpg

    if os.path.exists(img_path):
        # Process OCR and check rules
        ocr_results = extract_ocr_from_xml(img_path, xml_path)
        missing = check_missing_fields(ocr_results)
        
        # Simple rule: If fields are missing, flag as forged (1)
        is_suspicious = 1 if len(missing) > 0 else 0
        predictions.append(is_suspicious)
        
        print(f"Processed {xml_base}: {'Suspicious' if is_suspicious else 'Authentic'}")
    else:
        print(f"Image for {xml_base} missing in val folder, skipping...")

print(f"\nFinal Heuristic Predictions: {predictions}")

Found 20 annotated pairs to process in ../data/val...
Processed X00016469612: Suspicious
Processed X00016469623: Suspicious
Processed X51005230648: Suspicious
Processed X51005230659: Suspicious
Processed X51005303661: Suspicious
Processed X51005447840: Suspicious
Processed X51005568913: Suspicious
Processed X51005605334: Suspicious
Processed X51005663317: Suspicious
Processed X51005676535: Suspicious
Processed X51005705759: Suspicious
Processed X51005711401: Suspicious
Processed X51005712021: Suspicious
Processed X51005715455: Suspicious
Processed X51005719856: Suspicious
Processed X51005719863: Suspicious
Processed X51005719898: Suspicious
Processed X51005719914: Suspicious
Processed X51005724621: Suspicious
Processed X51005742068: Suspicious

Final Heuristic Predictions: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [3]:
%run ../src/train_classical.py

Extracting features for Classical ML...
Training RandomForest on 744 samples...
Success: Saved models/random_forest.joblib


In [4]:
%run ../src/train_deep.py

Starting CNN Baseline training...
Epoch 1/3 | Loss: 0.7037
Epoch 2/3 | Loss: 0.9118
Epoch 3/3 | Loss: 0.6439
Success: Saved models/cnn_baseline.pth
